# Deep Hedging — Phase 4, brique 5 : le pricer Heston semi-analytique

Pour couvrir avec d'autres options (étape suivante), il faut **valoriser une option à chaque date** le long de la trajectoire. On implémente donc le pricer Heston par **fonction caractéristique** (les équations de Riccati du document théorique), et on le valide de deux façons : contre le Monte-Carlo, et en vérifiant qu'il génère bien un **smile** à skew descendant.

In [ ]:
import numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq
from scipy.stats import norm
import matplotlib.pyplot as plt

## Fonction caractéristique et prix par inversion de Fourier

In [ ]:
def heston_cf(phi, S0, v0, r, kappa, theta, xi, rho, T):
    """Fonction caractéristique de ln S_T, forme à deux probabilités (Heston 1993)."""
    out = []
    for u, b in [(0.5, kappa - rho*xi), (-0.5, kappa)]:
        d = np.sqrt((rho*xi*1j*phi - b)**2 - xi**2*(2*u*1j*phi - phi**2))
        g = (b - rho*xi*1j*phi + d)/(b - rho*xi*1j*phi - d)
        C = r*1j*phi*T + (kappa*theta/xi**2)*((b - rho*xi*1j*phi + d)*T - 2*np.log((1-g*np.exp(d*T))/(1-g)))
        D = (b - rho*xi*1j*phi + d)/xi**2 * ((1-np.exp(d*T))/(1-g*np.exp(d*T)))
        out.append(np.exp(C + D*v0 + 1j*phi*np.log(S0)))
    return out

def heston_call(S0, v0, r, kappa, theta, xi, rho, T, K):
    def integ(phi, i):
        f = heston_cf(phi, S0, v0, r, kappa, theta, xi, rho, T)[i]
        return (np.exp(-1j*phi*np.log(K))*f/(1j*phi)).real
    P1 = 0.5 + quad(integ, 1e-8, 200, args=(0,), limit=200)[0]/np.pi
    P2 = 0.5 + quad(integ, 1e-8, 200, args=(1,), limit=200)[0]/np.pi
    return S0*P1 - K*np.exp(-r*T)*P2

## Validation 1 : contre le Monte-Carlo

In [ ]:
S0, v0, r = 100., 0.04, 0.02
kappa, theta, xi, rho, T = 2.0, 0.04, 0.3, -0.7, 1.0
rng = np.random.default_rng(0)
def sim_ST(m, n=252):
    dt=T/n; S=np.full(m,S0); v=np.full(m,v0)
    for _ in range(n):
        Z1=rng.standard_normal(m); Z2=rho*Z1+np.sqrt(1-rho**2)*rng.standard_normal(m); vk=np.maximum(v,0)
        S=S*np.exp((r-0.5*vk)*dt+np.sqrt(vk)*np.sqrt(dt)*Z1); v=np.maximum(v+kappa*(theta-vk)*dt+xi*np.sqrt(vk)*np.sqrt(dt)*Z2,0)
    return S
ST=sim_ST(400_000)
for K in [90,100,110]:
    cf=heston_call(S0,v0,r,kappa,theta,xi,rho,T,K); mc=np.exp(-r*T)*np.mean(np.maximum(ST-K,0))
    print(f"K={K}: CF={cf:.4f}  MC={mc:.4f}")

## Validation 2 : le smile généré

On price une gamme de strikes, on inverse en vol implicite Black-Scholes, et on trace. Heston avec ρ=−0.7 produit un **skew descendant**, exactement le smile des actions. C'est la preuve que le modèle fait ce qu'on attend.

In [ ]:
def bs_call(S,K,T,r,s):
    d1=(np.log(S/K)+(r+0.5*s**2)*T)/(s*np.sqrt(T)); d2=d1-s*np.sqrt(T)
    return S*norm.cdf(d1)-K*np.exp(-r*T)*norm.cdf(d2)
Ks=np.linspace(80,120,25)
ivs=[brentq(lambda s: bs_call(S0,K,T,r,s)-heston_call(S0,v0,r,kappa,theta,xi,rho,T,K),1e-3,2.0) for K in Ks]
plt.figure(figsize=(7,4.6))
plt.plot(Ks,100*np.array(ivs),"o-",color="navy"); plt.axvline(S0,color="grey",ls=":")
plt.axhline(100*np.sqrt(theta),color="crimson",ls="--",label="√θ = 20%")
plt.xlabel("strike K"); plt.ylabel("vol implicite (%)"); plt.title("Smile généré par Heston (ρ=−0.7)")
plt.legend(); plt.tight_layout(); plt.show()

## Utilité pour la suite

Ce pricer donne le prix d'une option pour tout `(S, v, tau)`. À l'étape 2, on l'utilisera pour valoriser l'**option de couverture** à chaque date de la trajectoire. Comme l'appeler des milliers de fois dans la boucle d'entraînement serait lent, on le **tabulera** (ou on entraînera un petit réseau à l'approcher) pour un accès rapide et différentiable.